# FGAI (ICT4403) Assignment 3: English → Indonesian Machine Translation

Four systems compared on the **same** test set:

| Model | Architecture | Adaptation |
|---|---|---|
| Basic Transformer (~20M) | encoder–decoder | trained from scratch |
| mT5-small (~300M) | encoder–decoder | full fine-tuning |
| Llama-3.2-1B-Instruct | decoder-only | 3-shot prompting + LoRA fine-tuning |
| Gemma-3-1B-it | decoder-only | 3-shot prompting + LoRA fine-tuning |

**Dataset:** OPUS-100 (en-id) — https://huggingface.co/datasets/Helsinki-NLP/opus-100

Run on a Colab T4 GPU. Set `Runtime → Change runtime type → T4 GPU` before running.
Llama and Gemma are loaded from the ungated `unsloth/` mirrors (same weights as the official Meta and Google releases), so no Hugging Face login is needed.

In [ ]:
!pip -q install -U transformers datasets sacrebleu sentencepiece peft accelerate tabulate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
import os, re, time, json, random, math
import numpy as np, torch, torch.nn as nn
from datasets import load_dataset
import sacrebleu

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    print("WARNING: no GPU found. Switch the runtime to a T4 GPU, this will be far too slow on CPU.")

# ---- experiment size (reduce these if you run out of time or memory) ----
N_TRAIN, N_VAL, N_TEST = 50_000, 2_000, 1_000   # from-scratch Transformer and mT5 use all of N_TRAIN
N_LLM_TRAIN   = 10_000                          # LoRA fine-tuning subset for Llama/Gemma (time budget)
N_PROMPT_EVAL = 200                             # test sentences used for the few-shot baselines
N_TEMP_EVAL   = 200                             # test sentences used for the temperature analysis
TEMPS = [0.0, 0.5, 1.0]                         # 0.0 = greedy
MAX_WORDS = 50

LLAMA_ID = "unsloth/Llama-3.2-1B-Instruct"   # ungated mirror of meta-llama/Llama-3.2-1B-Instruct
GEMMA_ID = "unsloth/gemma-3-1b-it"            # ungated mirror of google/gemma-3-1b-it
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
RESULTS = {}

In [ ]:
from huggingface_hub import notebook_login
# notebook_login()   # NOT needed: the unsloth mirrors are ungated. Uncomment only if a download asks for a token.

## 1. Dataset: description, preprocessing and splits
OPUS-100 is an English-centric multilingual parallel corpus sampled from OPUS (subtitles, web text, software
localisation, etc.). We use the `en-id` pair and its own train / validation / test splits, then:
1. drop empty pairs,
2. keep pairs of 3–50 words on both sides,
3. drop pairs whose length ratio is above 2.5 (usually misaligned),
4. de-duplicate on the English side,
5. shuffle with a fixed seed and take a manageable subset.

In [ ]:
raw = load_dataset("Helsinki-NLP/opus-100", "en-id")
print({k: len(v) for k, v in raw.items()})

def clean(split, n, seed):
    seen, rows = set(), []
    for ex in split:
        en, idn = ex["translation"]["en"].strip(), ex["translation"]["id"].strip()
        if not en or not idn: continue
        le, li = len(en.split()), len(idn.split())
        if not (3 <= le <= MAX_WORDS and 3 <= li <= MAX_WORDS): continue
        if max(le, li) / min(le, li) > 2.5: continue
        if en in seen: continue
        seen.add(en); rows.append((en, idn))
    random.Random(seed).shuffle(rows)
    return rows[:n]

train = clean(raw["train"], N_TRAIN, 1)
val   = clean(raw["validation"], N_VAL, 2)
test  = clean(raw["test"], N_TEST, 3)
test_src, test_ref = [e for e, _ in test], [i for _, i in test]

# leakage check: remove any test sentence that also appears in train
train_en = {e for e, _ in train}
overlap = sum(e in train_en for e in test_src)
print(f"train={len(train)} val={len(val)} test={len(test)} | test sentences also in train: {overlap}")

def avg_len(rows, k): return round(float(np.mean([len(r[k].split()) for r in rows])), 2)
stats = {s: {"pairs": len(r), "avg_en_words": avg_len(r, 0), "avg_id_words": avg_len(r, 1)}
         for s, r in [("train", train), ("val", val), ("test", test)]}
print(json.dumps(stats, indent=2))
json.dump(stats, open(f"{OUT}/dataset_stats.json", "w"), indent=2)
for en, idn in train[:5]: print("EN:", en, "\nID:", idn, "\n")

{'test': 2000, 'train': 1000000, 'validation': 2000}
train=50000 val=1708 test=1000 | test sentences also in train: 0
{
  "train": {
    "pairs": 50000,
    "avg_en_words": 8.01,
    "avg_id_words": 7.08
  },
  "val": {
    "pairs": 1708,
    "avg_en_words": 8.24,
    "avg_id_words": 7.13
  },
  "test": {
    "pairs": 1000,
    "avg_en_words": 8.13,
    "avg_id_words": 7.01
  }
}
EN: In whatever you may differ, [O believers] the verdict thereon rests with God. 
ID: Keputusan penyelesaian masalah keimanan dan kekafiran yang kalian perselisihkan itu berada di tangan Allah. Dan Dia sudah menjelaskan hal itu. 

EN: What do you mean, he hasn't done anything wrong? 
ID: Kenapa dia tak bersalah? 

EN: You went to him, didn't you? 
ID: Kau menemuinya, kan? 

EN: We're just going to leave him? 
ID: Kita tinggalkan dia begitu saja? 

EN: - I got it. - Andy's got it. 
ID: Kutangani. / Andy menanganinya. 



## 2. Shared helpers (metrics, timing, results store, probe sentences)

In [ ]:
HYP, TEMP, PROBE_OUT = {}, {}, {}

PROBES = [
    ("gender bias",    "He is a nurse and she is an engineer."),
    ("gender bias",    "The doctor told the nurse that she would be late."),
    ("gender bias",    "The CEO and his secretary arrived at the meeting."),
    ("numbers",        "The meeting is on 14 March at 3:30 pm in room 207."),
    ("numbers",        "The medicine costs $45.50 and must be taken twice daily."),
    ("safety-critical","Take two tablets after meals and never exceed four tablets in a day."),
    ("cultural",       "We celebrate Eid with family and friends."),
    ("cultural",       "She wore a hijab to her job interview."),
    ("cultural",       "Nyepi is the Balinese day of silence."),
    ("idiom",          "It is raining cats and dogs today."),
    ("idiom",          "He kicked the bucket last year."),
    ("sensitive",      "The patient was diagnosed with HIV and needs to start treatment."),
    ("sensitive",      "She said she felt hopeless and did not want to go on."),
    ("named entities", "Dr. Smith will fly from London to Jakarta on Monday."),
]
PROBE_SRC = [s for _, s in PROBES]

def sync():
    if device == "cuda": torch.cuda.synchronize()

def evaluate(hyps, refs):
    return {"BLEU": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 2),
            "chrF": round(sacrebleu.corpus_chrf(hyps, [refs]).score, 2)}

def count_params(m): return sum(p.numel() for p in m.parameters())

def flags(src, hyp):
    # cheap heuristics for hallucination-suspect outputs (NOT a proper hallucination detector)
    f = []
    ns = set(re.findall(r"\d+", src)); nh = set(re.findall(r"\d+", hyp))
    if ns != nh: f.append("number-mismatch")
    ratio = len(hyp.split()) / max(1, len(src.split()))
    if ratio > 2.0: f.append("too-long")
    if ratio < 0.5: f.append("too-short")
    if not hyp.strip(): f.append("empty")
    return f

def save_all():
    json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=2)
    json.dump(TEMP, open(f"{OUT}/temperature.json", "w"), indent=2)
    json.dump({"probes": PROBES, "outputs": PROBE_OUT}, open(f"{OUT}/probe_outputs.json", "w"),
              indent=2, ensure_ascii=False)

def temp_analysis(name, fn):
    s, r = test_src[:N_TEMP_EVAL], test_ref[:N_TEMP_EVAL]
    TEMP[name] = {}
    for T in TEMPS:
        torch.manual_seed(SEED)
        TEMP[name][str(T)] = evaluate(fn(s, T), r)
    print(name, "temperature analysis:", TEMP[name])

def run_all(name, fn, params_total, params_trainable, train_time_s):
    # fn(list_of_sources, temperature) -> list_of_translations
    sync(); t0 = time.time()
    hyps = fn(test_src, 0.0)
    sync(); dt = time.time() - t0
    HYP[name] = hyps
    RESULTS[name] = {**evaluate(hyps, test_ref), "n_eval": len(test_src),
                     "params_total": params_total, "params_trainable": params_trainable,
                     "train_time_s": round(train_time_s, 1), "infer_time_s": round(dt, 1),
                     "ms_per_sent": round(1000 * dt / len(test_src), 1)}
    print(name, RESULTS[name])
    temp_analysis(name, fn)
    PROBE_OUT[name] = fn(PROBE_SRC, 0.0)
    save_all()

## 3. Model 1: basic Transformer (encoder–decoder), trained from scratch
Joint SentencePiece BPE vocabulary (8k) over English and Indonesian, sinusoidal positions, tied embeddings,
3 encoder + 3 decoder layers, d_model 512, 8 heads, FFN 1024. Label smoothing 0.1, inverse-sqrt LR schedule,
early stopping on validation loss.

In [ ]:
import sentencepiece as spm
from torch.nn.utils.rnn import pad_sequence

with open("spm_train.txt", "w") as f:
    for en, idn in train: f.write(en + "\n" + idn + "\n")
spm.SentencePieceTrainer.train(input="spm_train.txt", model_prefix="spm_enid", vocab_size=8000,
                               model_type="bpe", character_coverage=1.0,
                               pad_id=0, unk_id=1, bos_id=2, eos_id=3)
sp = spm.SentencePieceProcessor(model_file="spm_enid.model")
PAD, UNK, BOS, EOS = 0, 1, 2, 3
MAXTOK = 80

def enc_pair(en, idn):
    s = sp.encode(en)[:MAXTOK - 1] + [EOS]
    t = [BOS] + sp.encode(idn)[:MAXTOK - 2] + [EOS]
    return torch.tensor(s), torch.tensor(t)

tr_data = [enc_pair(*p) for p in train]
va_data = [enc_pair(*p) for p in val]

class PosEnc(nn.Module):
    def __init__(self, d, maxlen=256):
        super().__init__()
        pe = torch.zeros(maxlen, d); pos = torch.arange(maxlen).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class TransformerMT(nn.Module):
    def __init__(self, vocab, d=512, heads=8, layers=3, ff=1024, drop=0.1):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.pos = PosEnc(d)
        self.tr = nn.Transformer(d, heads, layers, layers, ff, drop, batch_first=True)
        self.out = nn.Linear(d, vocab, bias=False)
        self.out.weight = self.emb.weight          # tied embeddings
    def encode(self, src):
        mask = src == PAD
        x = self.pos(self.emb(src) * math.sqrt(self.d))
        return self.tr.encoder(x, src_key_padding_mask=mask), mask
    def decode(self, tgt, mem, mem_mask):
        T = tgt.size(1)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=tgt.device)
        x = self.pos(self.emb(tgt) * math.sqrt(self.d))
        h = self.tr.decoder(x, mem, tgt_mask=causal, tgt_key_padding_mask=(tgt == PAD),
                            memory_key_padding_mask=mem_mask)
        return self.out(h)
    def forward(self, src, tgt_in):
        mem, m = self.encode(src)
        return self.decode(tgt_in, mem, m)

def batches(data, bs, shuffle):
    idx = list(range(len(data)))
    if shuffle: random.shuffle(idx)
    for i in range(0, len(idx), bs):
        b = [data[j] for j in idx[i:i + bs]]
        yield (pad_sequence([x[0] for x in b], batch_first=True, padding_value=PAD).to(device),
               pad_sequence([x[1] for x in b], batch_first=True, padding_value=PAD).to(device))

tmodel = TransformerMT(sp.get_piece_size()).to(device)
T_PARAMS = count_params(tmodel)
print(f"Transformer parameters: {T_PARAMS/1e6:.1f}M")

Transformer parameters: 19.9M


In [ ]:
EPOCHS_T, BS_T, PEAK_LR, WARM, PATIENCE = 15, 64, 7e-4, 1500, 3
opt = torch.optim.Adam(tmodel.parameters(), lr=PEAK_LR, betas=(0.9, 0.98), eps=1e-9)
crit = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)
amp = (device == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=amp)

def loss_fn(src, tgt):
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
        logits = tmodel(src, tgt[:, :-1])
        return crit(logits.reshape(-1, logits.size(-1)).float(), tgt[:, 1:].reshape(-1))

step, best, bad, best_state = 0, 1e9, 0, None
curves = {"train": [], "val": []}
sync(); t0 = time.time()
for ep in range(EPOCHS_T):
    tmodel.train(); tl, n = 0.0, 0
    for src, tgt in batches(tr_data, BS_T, True):
        step += 1
        for g in opt.param_groups: g["lr"] = PEAK_LR * min(step / WARM, math.sqrt(WARM / step))
        loss = loss_fn(src, tgt)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(tmodel.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tl += loss.item(); n += 1
    tmodel.eval(); vl, vn = 0.0, 0
    with torch.no_grad():
        for src, tgt in batches(va_data, 128, False):
            vl += loss_fn(src, tgt).item(); vn += 1
    curves["train"].append(tl / n); curves["val"].append(vl / vn)
    print(f"epoch {ep+1:02d} train {tl/n:.3f} val {vl/vn:.3f}")
    if vl / vn < best:
        best, bad = vl / vn, 0
        best_state = {k: v.detach().clone() for k, v in tmodel.state_dict().items()}
    else:
        bad += 1
        if bad >= PATIENCE: print("early stopping"); break
sync(); T_TRAIN_TIME = time.time() - t0
tmodel.load_state_dict(best_state)
print(f"training time: {T_TRAIN_TIME/60:.1f} min")
json.dump(curves, open(f"{OUT}/transformer_curves.json", "w"))

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


epoch 01 train 26.979 val 10.412
epoch 02 train 9.364 val 6.844
epoch 03 train 6.709 val 6.083
epoch 04 train 5.822 val 5.696
epoch 05 train 5.401 val 5.379
epoch 06 train 5.107 val 5.183
epoch 07 train 4.872 val 5.012
epoch 08 train 4.679 val 4.928
epoch 09 train 4.519 val 4.835
epoch 10 train 4.380 val 4.772
epoch 11 train 4.260 val 4.740
epoch 12 train 4.158 val 4.699
epoch 13 train 4.065 val 4.687
epoch 14 train 3.983 val 4.666
epoch 15 train 3.909 val 4.655
training time: 11.4 min


In [ ]:
@torch.no_grad()
def tr_translate(srcs, temperature=0.0, bs=64, maxlen=100):
    tmodel.eval(); outs = []
    for i in range(0, len(srcs), bs):
        batch = srcs[i:i + bs]
        ids = [torch.tensor(sp.encode(x)[:MAXTOK - 1] + [EOS]) for x in batch]
        src = pad_sequence(ids, batch_first=True, padding_value=PAD).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            mem, m = tmodel.encode(src)
            ys = torch.full((len(batch), 1), BOS, dtype=torch.long, device=device)
            done = torch.zeros(len(batch), dtype=torch.bool, device=device)
            for _ in range(maxlen):
                logits = tmodel.decode(ys, mem, m)[:, -1].float()
                if temperature <= 0:
                    nxt = logits.argmax(-1)
                else:
                    nxt = torch.multinomial(torch.softmax(logits / temperature, -1), 1).squeeze(1)
                nxt = nxt.masked_fill(done, PAD)
                ys = torch.cat([ys, nxt[:, None]], 1)
                done |= nxt == EOS
                if done.all(): break
        for row in ys[:, 1:].tolist():
            row = [t for t in row if t != PAD]
            if EOS in row: row = row[:row.index(EOS)]
            outs.append(sp.decode(row))
    return outs

run_all("Transformer (from scratch)", tr_translate, T_PARAMS, T_PARAMS, T_TRAIN_TIME)
del opt; torch.cuda.empty_cache()

Transformer (from scratch) {'BLEU': 9.1, 'chrF': 29.34, 'n_eval': 1000, 'params_total': 19869696, 'params_trainable': 19869696, 'train_time_s': 686.5, 'infer_time_s': 5.3, 'ms_per_sent': 5.3}
Transformer (from scratch) temperature analysis: {'0.0': {'BLEU': 8.12, 'chrF': 28.31}, '0.5': {'BLEU': 8.08, 'chrF': 28.57}, '1.0': {'BLEU': 2.84, 'chrF': 24.36}}


## 4. Model 2: mT5-small (encoder–decoder), full fine-tuning
Text-to-text with the prefix `translate English to Indonesian: `. fp32 is used on purpose because mT5 is
known to become unstable in fp16.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, get_linear_schedule_with_warmup

import gc, sys
for _v in ["tmodel", "best_state", "opt", "scaler", "mt5", "m_opt", "sched", "loss", "enc", "lab"]:
    if _v in globals(): del globals()[_v]
sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory in use before mT5: {torch.cuda.memory_allocated()/1e9:.2f} GB")

MT5_ID = "google/mt5-small"
mt5_tok = AutoTokenizer.from_pretrained(MT5_ID)
mt5 = AutoModelForSeq2SeqLM.from_pretrained(MT5_ID).to(device)
PREFIX = "translate English to Indonesian: "
M_PARAMS = count_params(mt5)
print(f"mT5-small parameters: {M_PARAMS/1e6:.1f}M")

def mt5_batch(pairs):
    enc = mt5_tok([PREFIX + e for e, _ in pairs], max_length=64, truncation=True, padding=True, return_tensors="pt")
    lab = mt5_tok(text_target=[i for _, i in pairs], max_length=64, truncation=True, padding=True,
                  return_tensors="pt")["input_ids"]
    lab[lab == mt5_tok.pad_token_id] = -100
    return {k: v.to(device) for k, v in enc.items()}, lab.to(device)

EPOCHS_M, BS_M, ACCUM_M, LR_M = 3, 16, 2, 5e-4   # effective batch 32; the 250k vocab makes big batches OOM
m_opt = torch.optim.AdamW(mt5.parameters(), lr=LR_M, weight_decay=0.01)
total_steps = EPOCHS_M * math.ceil(len(train) / (BS_M * ACCUM_M))
sched = get_linear_schedule_with_warmup(m_opt, int(0.03 * total_steps), total_steps)

sync(); t0 = time.time()
for ep in range(EPOCHS_M):
    mt5.train(); order = list(range(len(train))); random.shuffle(order); tl, n = 0.0, 0
    for i in range(0, len(order), BS_M):
        enc, lab = mt5_batch([train[j] for j in order[i:i + BS_M]])
        loss = mt5(**enc, labels=lab).loss / ACCUM_M
        loss.backward(); tl += loss.item() * ACCUM_M; n += 1
        if n % ACCUM_M == 0:
            nn.utils.clip_grad_norm_(mt5.parameters(), 1.0)
            m_opt.step(); sched.step(); m_opt.zero_grad(set_to_none=True)
    mt5.eval(); vl, vn = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(val), 32):
            enc, lab = mt5_batch(val[i:i + 32]); vl += mt5(**enc, labels=lab).loss.item(); vn += 1
    print(f"epoch {ep+1} train {tl/n:.3f} val {vl/vn:.3f}")
sync(); M_TRAIN_TIME = time.time() - t0
print(f"training time: {M_TRAIN_TIME/60:.1f} min")

@torch.no_grad()
def mt5_translate(srcs, T=0.0, bs=64):
    mt5.eval(); outs = []
    for i in range(0, len(srcs), bs):
        enc = mt5_tok([PREFIX + x for x in srcs[i:i + bs]], max_length=96, truncation=True,
                      padding=True, return_tensors="pt").to(device)
        kw = dict(do_sample=True, temperature=T, top_k=0) if T > 0 else dict(do_sample=False, num_beams=1)
        out = mt5.generate(**enc, max_new_tokens=96, **kw)
        outs += mt5_tok.batch_decode(out, skip_special_tokens=True)
    return outs

run_all("mT5-small (full fine-tune)", mt5_translate, M_PARAMS, M_PARAMS, M_TRAIN_TIME)
del mt5, m_opt, sched; torch.cuda.empty_cache()

GPU memory in use before mT5: 2.43 GB


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


mT5-small parameters: 300.2M
epoch 1 train 3.839 val 2.131
epoch 2 train 2.296 val 1.993
epoch 3 train 2.100 val 1.940
training time: 50.3 min
mT5-small (full fine-tune) {'BLEU': 22.74, 'chrF': 45.49, 'n_eval': 1000, 'params_total': 300176768, 'params_trainable': 300176768, 'train_time_s': 3019.4, 'infer_time_s': 13.5, 'ms_per_sent': 13.5}
mT5-small (full fine-tune) temperature analysis: {'0.0': {'BLEU': 20.87, 'chrF': 44.67}, '0.5': {'BLEU': 19.93, 'chrF': 43.42}, '1.0': {'BLEU': 13.57, 'chrF': 36.68}}


## 5. Models 3 and 4: Llama-3.2-1B and Gemma-3-1B (decoder-only)
For each model we run (a) a **3-shot prompting** baseline on the first `N_PROMPT_EVAL` test sentences, then
(b) **LoRA fine-tuning** on `N_LLM_TRAIN` training pairs (loss only on the Indonesian side), followed by merging
the adapter into the base weights so inference speed is comparable with the other models.

Note the comparison is not perfectly balanced: the LLMs see 10k training pairs versus 50k for the Transformer
and mT5, because of the time budget. This is stated in the README.

In [ ]:
from peft import LoraConfig, get_peft_model

def build_prompt(en):
    return f"Translate the following English sentence into Indonesian.\nEnglish: {en}\nIndonesian:"

FEWSHOT = train[:3]
def fewshot_prompt(en):
    shots = "".join(f"English: {a}\nIndonesian: {b}\n\n" for a, b in FEWSHOT)
    return "Translate English to Indonesian.\n\n" + shots + f"English: {en}\nIndonesian:"

@torch.no_grad()
def llm_translate(model, tok, srcs, prompt_fn, temperature=0.0, bs=16, max_new=96):
    model.eval(); tok.padding_side = "left"; outs = []
    for i in range(0, len(srcs), bs):
        enc = tok([prompt_fn(x) for x in srcs[i:i + bs]], return_tensors="pt", padding=True).to(device)
        kw = dict(do_sample=True, temperature=temperature, top_p=1.0, top_k=0) if temperature > 0 else dict(do_sample=False)
        out = model.generate(**enc, max_new_tokens=max_new, pad_token_id=tok.pad_token_id, **kw)
        for row in out[:, enc["input_ids"].shape[1]:]:
            txt = tok.decode(row, skip_special_tokens=True).strip()
            outs.append(txt.split("\n")[0].strip())
    return outs

def run_llm(label, model_id, dtype, epochs=1, bs=4, accum=4, lr=2e-4, maxlen=192):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to(device)
    base_total = count_params(model)

    # (a) few-shot prompting baseline
    sync(); t0 = time.time()
    h = llm_translate(model, tok, test_src[:N_PROMPT_EVAL], fewshot_prompt, 0.0)
    sync(); dt = time.time() - t0
    name = f"{label} (3-shot prompting)"
    HYP[name] = h
    RESULTS[name] = {**evaluate(h, test_ref[:N_PROMPT_EVAL]), "n_eval": N_PROMPT_EVAL,
                     "params_total": base_total, "params_trainable": 0, "train_time_s": 0.0,
                     "infer_time_s": round(dt, 1), "ms_per_sent": round(1000 * dt / N_PROMPT_EVAL, 1)}
    print(name, RESULTS[name]); save_all()

    # (b) LoRA fine-tuning
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, cfg)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{label}: trainable LoRA params {trainable/1e6:.2f}M of {base_total/1e6:.0f}M")

    def encode(en, idn):
        p = tok(build_prompt(en))["input_ids"]
        t = tok(" " + idn, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
        return (p + t)[:maxlen], ([-100] * len(p) + t)[:maxlen]
    data = [encode(en, idn) for en, idn in train[:N_LLM_TRAIN]]

    def collate(b):
        L = max(len(x[0]) for x in b)
        ids = torch.full((len(b), L), tok.pad_token_id); lab = torch.full((len(b), L), -100); att = torch.zeros((len(b), L), dtype=torch.long)
        for k, (i, l) in enumerate(b):
            ids[k, :len(i)] = torch.tensor(i); lab[k, :len(l)] = torch.tensor(l); att[k, :len(i)] = 1
        return ids.to(device), att.to(device), lab.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    o = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    total = epochs * math.ceil(len(data) / (bs * accum))
    sch = get_linear_schedule_with_warmup(o, int(0.05 * total), total)
    use_amp = dtype in (torch.float16, torch.bfloat16)
    scaler = torch.amp.GradScaler("cuda", enabled=(dtype == torch.float16))
    model.train(); sync(); t0 = time.time(); micro = 0; run_loss = 0.0
    for ep in range(epochs):
        order = list(range(len(data))); random.shuffle(order)
        for i in range(0, len(order), bs):
            ids, att, lab = collate([data[j] for j in order[i:i + bs]])
            with torch.autocast("cuda", dtype=dtype, enabled=use_amp):
                loss = model(input_ids=ids, attention_mask=att, labels=lab).loss / accum
            scaler.scale(loss).backward(); micro += 1; run_loss += loss.item() * accum
            if micro % accum == 0:
                scaler.unscale_(o); nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(o); scaler.update(); sch.step(); o.zero_grad(set_to_none=True)
            if micro % 200 == 0:
                print(f"{label} micro-step {micro} loss {run_loss/200:.3f}"); run_loss = 0.0
    sync(); train_time = time.time() - t0
    print(f"{label} LoRA training time: {train_time/60:.1f} min")

    model = model.merge_and_unload(); model.eval()
    run_all(f"{label} (LoRA fine-tune)", lambda s, T: llm_translate(model, tok, s, build_prompt, T),
            count_params(model), trainable, train_time)
    del model, o, sch; torch.cuda.empty_cache()

CC = torch.cuda.get_device_capability()[0] if device == "cuda" else 0
LLAMA_DTYPE = torch.bfloat16 if CC >= 8 else torch.float16
GEMMA_DTYPE = torch.bfloat16 if CC >= 8 else torch.float32   # Gemma overflows in fp16
print(CC, LLAMA_DTYPE, GEMMA_DTYPE)

In [ ]:
run_llm("Llama-3.2-1B", LLAMA_ID, LLAMA_DTYPE)

In [ ]:
run_llm("Gemma-3-1B", GEMMA_ID, GEMMA_DTYPE)

## 6. Results: BLEU, chrF, time, parameters

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

rows = []
for name, r in RESULTS.items():
    sub = evaluate(HYP[name][:N_PROMPT_EVAL], test_ref[:N_PROMPT_EVAL])
    rows.append({"Model": name, "Eval sents": r["n_eval"], "BLEU": r["BLEU"], "chrF": r["chrF"],
                 f"BLEU@{N_PROMPT_EVAL}": sub["BLEU"], f"chrF@{N_PROMPT_EVAL}": sub["chrF"],
                 "Params (M)": round(r["params_total"] / 1e6, 1),
                 "Trainable (M)": round(r["params_trainable"] / 1e6, 2),
                 "Train time (min)": round(r["train_time_s"] / 60, 1),
                 "Infer time (s)": r["infer_time_s"], "ms/sent": r["ms_per_sent"]})
df = pd.DataFrame(rows)
open(f"{OUT}/results_table.md", "w").write(df.to_markdown(index=False))
df.to_csv(f"{OUT}/results_table.csv", index=False)
print(df.to_markdown(index=False))

main = [n for n in RESULTS if "prompting" not in n]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
x = np.arange(len(main)); w = 0.38
ax[0].bar(x - w/2, [RESULTS[n]["BLEU"] for n in main], w, label="BLEU")
ax[0].bar(x + w/2, [RESULTS[n]["chrF"] for n in main], w, label="chrF")
ax[0].set_xticks(x); ax[0].set_xticklabels([n.split(" (")[0] for n in main], rotation=20); ax[0].legend(); ax[0].set_title("Quality (test set)")
ax[1].bar(x, [RESULTS[n]["train_time_s"] / 60 for n in main]); ax[1].set_xticks(x)
ax[1].set_xticklabels([n.split(" (")[0] for n in main], rotation=20); ax[1].set_title("Training / fine-tuning time (min)")
ax[2].bar(x, [RESULTS[n]["ms_per_sent"] for n in main]); ax[2].set_xticks(x)
ax[2].set_xticklabels([n.split(" (")[0] for n in main], rotation=20); ax[2].set_title("Inference (ms / sentence)")
plt.tight_layout(); plt.savefig(f"{OUT}/comparison.png", dpi=150); plt.show()

plt.figure(figsize=(6, 4))
for n in main:
    plt.plot(TEMPS, [TEMP[n][str(T)]["BLEU"] for T in TEMPS], marker="o", label=n.split(" (")[0])
plt.xlabel("sampling temperature (0 = greedy)"); plt.ylabel(f"BLEU on first {N_TEMP_EVAL} test sentences")
plt.legend(); plt.title("Temperature analysis"); plt.tight_layout(); plt.savefig(f"{OUT}/temperature.png", dpi=150); plt.show()

tdf = pd.DataFrame([{"Model": n.split(" (")[0], "T": T, **TEMP[n][str(T)]} for n in main for T in TEMPS])
open(f"{OUT}/temperature_table.md", "w").write(tdf.to_markdown(index=False)); print(tdf.to_markdown(index=False))

In [ ]:
# generated samples for the README / report
with open(f"{OUT}/generated_samples.md", "w") as f:
    for k in range(20):
        f.write(f"### Example {k+1}\n- **Source:** {test_src[k]}\n- **Reference:** {test_ref[k]}\n")
        for n in main: f.write(f"- **{n}:** {HYP[n][k]}\n")
        f.write("\n")
pd.DataFrame({"source": test_src, "reference": test_ref, **{n: HYP[n] for n in main}}).to_csv(f"{OUT}/test_translations.csv", index=False)
print(open(f"{OUT}/generated_samples.md").read()[:3000])

## 7. Responsible and ethical AI analysis
Hand-written probe sentences (gender bias, numbers, safety-critical, cultural, idioms, sensitive content,
named entities) are translated by every fine-tuned model. The automatic flags are cheap heuristics
(number mismatch, extreme length ratio), not a hallucination detector. Indonesian writes decimals with a comma,
so "number-mismatch" on prices can be a false alarm. **Read the actual outputs yourself** and write the
discussion in the README from what you observe.

In [ ]:
lines = ["| Category | Source | " + " | ".join(n.split(" (")[0] for n in main) + " |", "|---|---|" + "---|" * len(main)]
for k, (cat, src) in enumerate(PROBES):
    cells_ = []
    for n in main:
        h = PROBE_OUT[n][k]; fl = flags(src, h)
        cells_.append(h.replace("|", "/") + (f" ⚠ {','.join(fl)}" if fl else ""))
    lines.append(f"| {cat} | {src} | " + " | ".join(cells_) + " |")
open(f"{OUT}/ethics_probe.md", "w").write("\n".join(lines))
print("\n".join(lines))

rates = {}
for n in main:
    fl = [flags(s, h) for s, h in zip(test_src, HYP[n])]
    rates[n] = {k: round(100 * sum(k in f for f in fl) / len(fl), 1) for k in ["number-mismatch", "too-long", "too-short", "empty"]}
json.dump(rates, open(f"{OUT}/flag_rates.json", "w"), indent=2)
print(pd.DataFrame(rates).T.to_markdown())

In [ ]:
!zip -rq outputs.zip outputs
from google.colab import files
files.download("outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
json.dump({"RESULTS": RESULTS, "HYP": HYP, "TEMP": TEMP, "PROBE_OUT": PROBE_OUT},
          open("outputs/state_after_transformer.json", "w"), ensure_ascii=False)
print("saved", list(RESULTS))

saved []


In [ ]:
st = json.load(open("outputs/state_after_transformer.json"))
RESULTS.update(st["RESULTS"]); HYP.update(st["HYP"]); TEMP.update(st["TEMP"]); PROBE_OUT.update(st["PROBE_OUT"])
print(list(RESULTS))

[]
